In [ ]:
import pandas as pd

# Path to data frame with WSIs
# df_path = "D:\DATA\with_snomed_category.csv"
# df_path = r"D:\DATA\abmil_exp2.csv"
# df_path = r"D:\DATA\abmil_exp3.csv"
# df_path = r"D:\DATA\abmil_inference_exp3.csv"
df_path = r"D:\DATA\abmil_inference_exp3_vers3.csv"

df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"
# checkpoint_path = r'D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0/fold_1_auc_0.8883.pt'
checkpoint_path = r'D:\NOTEBOOKS\Christine\abmil_checkpoints\abmil_hopt_full.pt'
# cache_path = r"D:\NOTEBOOKS\Christine\checkpoints\exp3_h-optimus-0\fold_1_inference_cache.pkl"
cache_path = r"D:\NOTEBOOKS\Christine\abmil_checkpoints\abmil_hopt_full_inference.pkl"
xml_dir = r"D:\NOTEBOOKS\Christine\exp3\xml"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
all_filenames = df_all["filename"].tolist()
print("Number of files: ", len(all_filenames))

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 2, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 3,
    'Resection Margin Not Free': 3, 
    'Proliferative/Pre-neoplastic Changes': 3, 
    'Benign Neoplasm': 3, 
    'Uncertain / Borderline Neoplasm': 3, 
    'In Situ Neoplasm': 3, 
    'Malignant Neoplasm': 3,
}

df_all["M_idx"] = df_all["M_category"].apply(
    lambda lst: [class_dict[x] for x in lst]
)

# 0: Normal
# 1: Other morphologies
# 2: Inflammation
# 3: Neoplastic Changes, Benign/Uncertain/Borderline, In Situ, Malignant Neoplasm 

In [ ]:
df_all['M_idx'] = df_all['M_idx'].apply(
    lambda x: max(x) if isinstance(x, list) else x
)

In [ ]:
print(df_all.head())

In [ ]:
from abmil_v2 import load_checkpoint

# Load and verify checkpoint
loaded_model, config, loaded_label_mapping = load_checkpoint(checkpoint_path)

print(f"\n✓ Checkpoint loaded successfully!")
print(f"\nConfig keys: {config.keys()}")
print(f"Config:\n{config}")
print(f"\nLabel mapping: {loaded_label_mapping}")

In [ ]:
from abmil_pipeline import ABMILInference, ABMILEvaluation

# Initialize inference
inference = ABMILInference(
    checkpoint_path=checkpoint_path,
    zarr_dir=zarr_dir,
    slides=all_filenames,
    cache_path=cache_path,
)

print(f"✓ Inference initialized")
print(f"  Label mapping: {inference.label_mapping}")
print(f"  Feature key: {inference.feature_key}")
print(f"  Tile key: {inference.tile_key}")

print('Classifier initialized for', len(all_filenames), 'slides')


In [ ]:
# Process all slides (this may take a while)
inference.process_slides()

In [ ]:
results_df = inference.results_dataframe()
print(f"Results for {len(results_df)} slides:\n")
print(results_df.head())

In [ ]:
# Save results to CSV
results_df.to_csv(results_csv, index=False)
print(f"\n✓ Results saved to: {results_csv}")

In [ ]:
from helper_functions import lists2tuples

df_all = lists2tuples(df_all)

In [ ]:
# Initialize evaluator if you have ground truth labels
evaluator = ABMILEvaluation(results_df, metadata_df = df_all, true_label_col="M_idx")

# Match predictions with ground truth
evaluator.match_true_labels(slide_id_col="filename", results_path_col="slide_path")
print(f"Matched {len(matched_df)} slides with ground truth")
print(matched_df.head())

In [ ]:
# Compute metrics
metrics = evaluator.compute_metrics()
print(f"\nMacro AUC: {metrics['auc']:.4f}")
print(f"Per-class AUC: {metrics['per_class_aucs']}")

# Confusion matrix and classification report
evaluator.assessment_report()

In [ ]:
# Plot precision-recall curves
evaluator.pr_curves()

In [ ]:
# Plot ROC curves for all classes
evaluator.roc_curves_all()

In [ ]:
group_metrics = evaluator.group_by_metrics("T_category")

In [ ]:
slide = all_filenames[0]  # Change index to select a different slide
inference.attention_heatmap(slide)

In [ ]:
from roi_selection import ROISelector

# Select top_k ROIs 
roi_selector = ROISelector(cache_path=cache_path, slide_path=slide, top_k=100)

In [ ]:
roi_selector.zoomed_view()

In [ ]:
roi_selector.tiles_to_cut()

In [ ]:
df_slide = df_all[df_all["filename"].str.endswith("12203427010101.mrxs", na=False)]
df_slide.iloc[0]

In [ ]:
# Preserve the notebook variables used by the later ROI/Napari cells
top_tiles_gdf = roi_selector.top_tiles_from_slide_data()
wsi = roi_selector.get_wsi()
sdata = roi_selector.get_sdata()
roi_polygons = roi_selector.napari_polygons()